# VGGT Point Cloud Comparison With And Without IMU Guidance

This notebook is designed for Google Colab free tier.

It does two runs on the same ADVIO sequence:

1. `VGGT baseline` on a small image set capped at 15 images
2. `IMU-guided VGGT` on the same sequence, where IMU is used to choose motion-informative frames before inference

Then it visualizes the point clouds from both runs.

Important note:
VGGT itself is still vision-only. In this notebook, IMU helps by guiding frame selection and providing pose priors for comparison, not by being injected into the VGGT transformer.

## 1. Mount Drive And Open The Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/drive/MyDrive/3d-reconstruction'
%cd $REPO_DIR

import os
print('Repo exists:', os.path.isdir(REPO_DIR))

## 2. Install Dependencies

In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt
!python -m pip install -q plotly huggingface_hub einops

## 3. Clone Official VGGT And Download Checkpoint

In [ ]:
import os, sys

VGGT_REPO = '/content/vggt'
CKPT_PATH = '/content/model_tracker_fixed_e20.pt'

if not os.path.exists(VGGT_REPO):
    !git clone https://github.com/facebookresearch/vggt.git $VGGT_REPO

if VGGT_REPO not in sys.path:
    sys.path.insert(0, VGGT_REPO)

if not os.path.exists(CKPT_PATH):
    !wget -q --show-progress https://huggingface.co/facebook/VGGT_tracker_fixed/resolve/main/model_tracker_fixed_e20.pt -O $CKPT_PATH

print('VGGT repo:', VGGT_REPO)
print('Checkpoint exists:', os.path.exists(CKPT_PATH))

## 4. Download One ADVIO Sequence

Sequence 15 is a calm office sequence and a good first test.

In [ ]:
import os

ADVIO_SEQ = '15'
ADVIO_ROOT = f'data/advio-{ADVIO_SEQ}'
ADVIO_ZIP = f'data/advio-{ADVIO_SEQ}.zip'
ADVIO_URL = f'https://zenodo.org/record/1476931/files/advio-{ADVIO_SEQ}.zip'

os.makedirs('data', exist_ok=True)
if not os.path.exists(ADVIO_ZIP):
    !wget -O $ADVIO_ZIP $ADVIO_URL
if not os.path.exists(ADVIO_ROOT):
    !unzip -q -o $ADVIO_ZIP -d data

!find $ADVIO_ROOT -maxdepth 2 | head -n 20

## 5. Prepare A Dataset With At Most 15 Baseline Images

Baseline uses up to 15 uniformly spaced frames from the sequence.
IMU-guided uses the same prepared sequence and keeps only motion-informative frames from those candidates.

In [ ]:
import math
from src.datasets.advio import load_advio_iphone_sequence

MAX_IMAGES = 15
ROT_THRESH_DEG = 8.0
TRANS_THRESH_M = 0.0
MIN_GAP = 1
PREP_ROOT = f'data/prepared_15/advio-{ADVIO_SEQ}'

seq = load_advio_iphone_sequence(ADVIO_ROOT)
every_nth = max(1, math.ceil(len(seq.frame_timestamps) / MAX_IMAGES))
print('Total frames in sequence:', len(seq.frame_timestamps))
print('Sampling every nth frame:', every_nth)

!python prepare_advio_vggt_dataset.py \
  --advio-root $ADVIO_ROOT \
  --output-root $PREP_ROOT \
  --every-nth-frame $every_nth \
  --max-frames $MAX_IMAGES \
  --rotation-thresh-deg $ROT_THRESH_DEG \
  --translation-thresh-m $TRANS_THRESH_M \
  --min-gap $MIN_GAP

!cat $PREP_ROOT/summary.json

## 6. Load VGGT

In [ ]:
import torch
from vggt.models.vggt import VGGT

device = 'cuda' if torch.cuda.is_available() else 'cpu'
cap = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16

print('Device:', device)
print('dtype:', DTYPE)

model = VGGT()
state_dict = torch.load(CKPT_PATH, map_location='cpu')
if 'model' in state_dict:
    state_dict = state_dict['model']
model.load_state_dict(state_dict, strict=False)
model = model.to(device).eval()
print('VGGT loaded.')

## 7. Helper Functions

In [ ]:
import time
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
import torchvision.transforms as T
from PIL import Image

IMG_SIZE = 518
transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def load_images_from_dir(img_dir):
    exts = {'.jpg', '.jpeg', '.png', '.JPG', '.PNG'}
    paths = sorted([p for p in Path(img_dir).iterdir() if p.suffix in exts])
    tensors = []
    pil_images = []
    for p in paths:
        img = Image.open(p).convert('RGB')
        pil_images.append(img)
        tensors.append(transform(img))
    return torch.stack(tensors), paths, pil_images

def run_vggt_on_dir(img_dir, output_pose_dir=None):
    images_tensor, img_paths, pil_images = load_images_from_dir(img_dir)
    images_input = images_tensor.unsqueeze(0).to(device, dtype=DTYPE)

    t0 = time.time()
    with torch.no_grad():
        with torch.cuda.amp.autocast(dtype=DTYPE):
            aggregated_tokens_list, ps_idx = model.aggregator(images_input)
            from vggt.utils.pose_enc import pose_encoding_to_extri_intri
            pose_enc = model.camera_head(aggregated_tokens_list)[-1]
            extrinsic, intrinsic = pose_encoding_to_extri_intri(pose_enc, images_input.shape[-2:])
            depth_map, depth_conf = model.depth_head(aggregated_tokens_list, images_input, ps_idx)
            from vggt.utils.geometry import unproject_depth_map_to_point_map
            point_map_unproj = unproject_depth_map_to_point_map(depth_map.squeeze(0), extrinsic.squeeze(0), intrinsic.squeeze(0))
    elapsed = time.time() - t0

    extrinsic_np = extrinsic.squeeze(0).float().cpu().numpy()
    intrinsic_np = intrinsic.squeeze(0).float().cpu().numpy()
    depth_np = depth_map.squeeze(0).float().cpu().numpy()
    depth_conf_np = depth_conf.squeeze(0).float().cpu().numpy()
    point_np = point_map_unproj.float().cpu().numpy()

    if output_pose_dir is not None:
        out = Path(output_pose_dir)
        out.mkdir(parents=True, exist_ok=True)
        for i, path in enumerate(img_paths):
            np.savetxt(out / f'{path.stem}_extrinsic.txt', extrinsic_np[i], fmt='%.8f')
            np.savetxt(out / f'{path.stem}_intrinsic.txt', intrinsic_np[i], fmt='%.8f')

    return {
        'img_paths': img_paths,
        'pil_images': pil_images,
        'extrinsic': extrinsic_np,
        'intrinsic': intrinsic_np,
        'depth': depth_np,
        'depth_conf': depth_conf_np,
        'point_map': point_np,
        'elapsed_sec': elapsed,
    }

def merged_point_cloud(result, conf_threshold=1.5, max_points=60000):
    points = result['point_map'].reshape(-1, 3)
    conf = result['depth_conf'].reshape(-1)
    mask = np.isfinite(points).all(axis=1) & (conf > conf_threshold)
    points = points[mask]
    if len(points) > max_points:
        idx = np.random.choice(len(points), size=max_points, replace=False)
        points = points[idx]
    return points

def show_point_cloud(points, title):
    fig = go.Figure(data=[
        go.Scatter3d(
            x=points[:, 0], y=points[:, 1], z=points[:, 2],
            mode='markers',
            marker=dict(size=1.5, opacity=0.7)
        )
    ])
    fig.update_layout(
        title=title,
        scene=dict(aspectmode='data'),
        height=700,
        margin=dict(l=0, r=0, t=40, b=0),
    )
    fig.show()

## 8. Run Baseline VGGT Inference And Visualize Point Cloud

In [ ]:
BASELINE_IMG_DIR = f'{PREP_ROOT}/baseline/images'
BASELINE_POSE_DIR = f'outputs/advio-{ADVIO_SEQ}/vggt_15/baseline/poses'

baseline_result = run_vggt_on_dir(BASELINE_IMG_DIR, BASELINE_POSE_DIR)
baseline_points = merged_point_cloud(baseline_result)

print('Baseline image count:', len(baseline_result['img_paths']))
print('Baseline inference time (s):', baseline_result['elapsed_sec'])
print('Baseline merged points:', len(baseline_points))

show_point_cloud(baseline_points, 'Baseline VGGT Point Cloud')

## 9. Run IMU-Guided VGGT Inference And Visualize Point Cloud

In [ ]:
IMU_IMG_DIR = f'{PREP_ROOT}/imu_guided/images'
IMU_POSE_DIR = f'outputs/advio-{ADVIO_SEQ}/vggt_15/imu_guided/poses'

imu_result = run_vggt_on_dir(IMU_IMG_DIR, IMU_POSE_DIR)
imu_points = merged_point_cloud(imu_result)

print('IMU-guided image count:', len(imu_result['img_paths']))
print('IMU-guided inference time (s):', imu_result['elapsed_sec'])
print('IMU-guided merged points:', len(imu_points))

show_point_cloud(imu_points, 'IMU-Guided VGGT Point Cloud')

## 10. Optional Quantitative Comparison Against ADVIO Ground Truth

In [ ]:
REPORT_JSON = f'outputs/analysis/advio-{ADVIO_SEQ}_vggt_15_benchmark.json'
PLOT_DIR = f'outputs/analysis/advio-{ADVIO_SEQ}_vggt_15'

!python benchmark_advio_vggt.py \
  --advio-root $ADVIO_ROOT \
  --baseline-frames-csv $PREP_ROOT/baseline/metadata/frame_times.csv \
  --baseline-pose-dir $BASELINE_POSE_DIR \
  --imu-guided-frames-csv $PREP_ROOT/imu_guided/metadata/selected_frames.csv \
  --imu-guided-pose-dir $IMU_POSE_DIR \
  --output-json $REPORT_JSON

!python visualize_advio_benchmark.py \
  --advio-root $ADVIO_ROOT \
  --baseline-frames-csv $PREP_ROOT/baseline/metadata/frame_times.csv \
  --baseline-pose-dir $BASELINE_POSE_DIR \
  --imu-guided-frames-csv $PREP_ROOT/imu_guided/metadata/selected_frames.csv \
  --imu-guided-pose-dir $IMU_POSE_DIR \
  --output-dir $PLOT_DIR

!cat $REPORT_JSON

## 11. Read The Tradeoff

What to look for:

- whether the IMU-guided run kept fewer than 15 images
- whether the point cloud stays visually coherent
- whether inference time drops
- whether pose error stays stable or gets worse
- whether the GPU proxies in the benchmark improve a lot while geometry stays acceptable